# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrinidhi-02/flyrank-machine-learning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will prioritize content that has a clear opportunity for improvement based on its observed performance and available signals. The baseline is a simple, explainable decision-support rule, not a prediction of guaranteed future results.

### Reason codes

- `REFRESH_REVIEW`: The observed content age suggests that a refresh may be worth reviewing.
- `CTR_REVIEW`: The observed click-through rate looks weak relative to the content's search position.
- `VOLUME_REVIEW`: The available performance volume suggests that the content may be worth investigating.

The score will rank items for review. These reason codes are suggestions for investigation, not proof that an item is underperforming.

In [42]:

import pandas as pd
from pathlib import Path

# Load the anonymized starter dataset from the repository
data_url = (
    "https://raw.githubusercontent.com/"
    "Shrinidhi-02/flyrank-machine-learning/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nAvailable columns:")
print(df.columns.tolist())



Dataset loaded successfully
Rows: 30000
Columns: 44

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [43]:
# Inspect the main signals we can use for the baseline

check_columns = [
    "content_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "impressions_last_30d",
    "impressions_prev_30d",
    "trend_direction",
]

print("Sample rows:")
display(df[check_columns].head(10))

print("\nMissing values:")
print(df[check_columns].isna().sum())

print("\nSummary statistics:")
display(df[[
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
]].describe())

Sample rows:


,content_id,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,impressions_last_30d,impressions_prev_30d,trend_direction
0,content_304f48230142,187,20,3803,29,0.76,10.6,578,987,down
1,content_a1fb4e703a9e,445,25,15320,7,0.05,20.3,2501,5915,down
2,content_9aa793d4d895,141,20,12581,11,0.09,36.5,2382,6089,down
3,content_331d6c4de07b,463,22,11751,58,0.49,6.2,3626,4206,stable
4,content_d99b7a2d90ca,263,14,19140,24,0.13,44.0,4211,6452,down
5,content_d4084a4bc775,147,20,3970,1,0.03,8.5,617,1009,down
6,content_9a34b442b552,90,20,20,0,0.00,7.0,1,13,down
7,content_a63219c6e95a,445,22,1724,1,0.06,21.2,636,632,stable
8,content_5e6c160719bc,90,20,32574,29,0.09,46.0,5696,13828,down
9,content_c27558df2b0c,257,104,1240,2,0.16,4.9,252,356,down



Missing values:
content_id                0
content_age_days          0
days_since_last_update    0
impressions_90d           0
clicks_90d                0
ctr                       0
avg_position              0
impressions_last_30d      0
impressions_prev_30d      0
trend_direction           0
dtype: int64

Summary statistics:


,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position
count,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000,30000.00000
mean,256.16780,46.098300,5200.366300,16.097333,0.510733,16.34238
std,132.70793,42.078709,16838.019547,75.076958,3.279162,15.21679
min,90.00000,1.000000,1.000000,0.000000,0.000000,0.00000
25%,132.00000,20.000000,81.000000,0.000000,0.000000,6.20000
50%,236.00000,20.000000,731.000000,1.000000,0.070000,10.80000
75%,333.00000,104.000000,3615.250000,7.000000,0.290000,22.30000
max,564.00000,373.000000,517715.000000,4178.000000,100.000000,245.00000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [44]:

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Load the dataset
# --------------------------------------------------

data_url = (
    "https://raw.githubusercontent.com/"
    "Shrinidhi-02/flyrank-machine-learning/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))


# --------------------------------------------------
# 2. Set thresholds for the baseline rule
# --------------------------------------------------

# Content in the oldest 25% by update age
refresh_threshold = df["days_since_last_update"].quantile(0.75)

# Content in the highest 25% by impressions
volume_threshold = df["impressions_90d"].quantile(0.75)

# Calculate median CTR only for valid top-10 positions
valid_top10 = (
    df["avg_position"].notna()
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 10)
    & df["ctr"].notna()
)

top10_ctr_median = df.loc[valid_top10, "ctr"].median()

print("\nThresholds")
print("Refresh threshold:", refresh_threshold)
print("Impression threshold:", volume_threshold)
print("Median CTR for valid top-10 results:", top10_ctr_median)


# --------------------------------------------------
# 3. Create the baseline signals
# --------------------------------------------------

scored = df.copy()

# Signal 1: content may need a freshness review
refresh_flag = (
    scored["days_since_last_update"].notna()
    & (scored["days_since_last_update"] >= refresh_threshold)
)

# Signal 2: CTR is low for a valid top-10 result
ctr_flag = (
    scored["avg_position"].notna()
    & (scored["avg_position"] > 0)
    & (scored["avg_position"] <= 10)
    & scored["ctr"].notna()
    & (scored["ctr"] < top10_ctr_median)
)

# Signal 3: content has high observed impression volume
volume_flag = (
    scored["impressions_90d"].notna()
    & (scored["impressions_90d"] >= volume_threshold)
)


# --------------------------------------------------
# 4. Calculate the baseline score
# --------------------------------------------------

# Refresh signal = 2 points
# Low CTR signal = 2 points
# High impression volume = 1 point

scored["score"] = (
    refresh_flag.astype(int) * 2
    + ctr_flag.astype(int) * 2
    + volume_flag.astype(int)
)

scored["score"] = scored["score"].astype(int)


# --------------------------------------------------
# 5. Assign reason codes and action labels
# --------------------------------------------------

scored["reason_code"] = "NO_STRONG_SIGNAL"
scored["action_label"] = "Manual review if needed"

# Start with the lowest-priority signal
scored.loc[
    volume_flag,
    ["reason_code", "action_label"]
] = [
    "VOLUME_REVIEW",
    "Review high-volume content"
]

# Refresh takes priority over volume
scored.loc[
    refresh_flag,
    ["reason_code", "action_label"]
] = [
    "REFRESH_REVIEW",
    "Assess content freshness"
]

# CTR takes priority over both other signals
scored.loc[
    ctr_flag,
    ["reason_code", "action_label"]
] = [
    "CTR_REVIEW",
    "Review title and snippet"
]


# --------------------------------------------------
# 6. Rank the content items
# --------------------------------------------------

# Sort by score, then impressions, then stable content ID.
# IDs are used only for tie-breaking, not as score features.

scored = scored.sort_values(
    by=["score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
    na_position="last"
).reset_index(drop=True)

scored.insert(0, "rank", range(1, len(scored) + 1))


# --------------------------------------------------
# 7. Create explanations for the Top 20
# --------------------------------------------------

top20 = scored.head(20).copy()


def explain_why(row):
    reasons = []

    if (
        pd.notna(row["days_since_last_update"])
        and row["days_since_last_update"] >= refresh_threshold
    ):
        reasons.append(
            "Update age is in the oldest 25% of the dataset"
        )

    if (
        pd.notna(row["avg_position"])
        and 0 < row["avg_position"] <= 10
        and pd.notna(row["ctr"])
        and row["ctr"] < top10_ctr_median
    ):
        reasons.append(
            "CTR is below the median for valid top-10 results"
        )

    if (
        pd.notna(row["impressions_90d"])
        and row["impressions_90d"] >= volume_threshold
    ):
        reasons.append(
            "Impressions are in the highest 25% of the dataset"
        )

    if reasons:
        return "; ".join(reasons)

    return "No qualifying signal; investigate the available data"


top20["why_selected"] = top20.apply(explain_why, axis=1)

top20["what_could_make_it_wrong"] = (
    "The signal may not reflect a real improvement opportunity; "
    "check search intent, data completeness, and business context "
    "before acting."
)


# --------------------------------------------------
# 8. Display Top 20 for review
# --------------------------------------------------

review_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action_label",
    "why_selected",
    "what_could_make_it_wrong"
]

print("\nTOP 20 CONTENT ITEMS FOR REVIEW")
display(top20[review_columns])

print("\nItems reviewed:", len(top20))


# --------------------------------------------------
# 9. Check score and reason-code distributions
# --------------------------------------------------

print("\nScore distribution:")
display(
    scored["score"]
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis("score")
    .to_frame("count")
)

print("\nReason-code counts:")
display(
    scored["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .to_frame("count")
)


# --------------------------------------------------
# 10. Export the complete ranked dataset
# --------------------------------------------------

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

scored.to_csv(output_path, index=False)

print("\nCSV saved successfully:", output_path)
print("Rows exported:", len(scored))
print("File size in bytes:", output_path.stat().st_size)


Dataset loaded successfully
Rows: 30000
Columns: 44

Thresholds
Refresh threshold: 104.0
Impression threshold: 3615.25
Median CTR for valid top-10 results: 0.15

TOP 20 CONTENT ITEMS FOR REVIEW


,rank,content_id,score,reason_code,action_label,why_selected,what_could_make_it_wrong
0,1,content_5fe46e04994d,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
1,2,content_36ff89c8214e,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
2,3,content_c8e9d6ab9013,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
3,4,content_a7427266c305,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
4,5,content_91652435f57a,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
5,6,content_f42eb861c6dd,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
6,7,content_97a86caf3a3d,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
7,8,content_c1fe78bc4e37,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
8,9,content_4a6607efcb46,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
9,10,content_4c76e9b13aea,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...



Items reviewed: 20

Score distribution:


,count
score,
5,474
4,1175
3,3352
2,8873
1,3674
0,12452



Reason-code counts:


,count
reason_code,
NO_STRONG_SIGNAL,12452
REFRESH_REVIEW,7442
CTR_REVIEW,6432
VOLUME_REVIEW,3674



CSV saved successfully: work/outputs/baseline_action_score.csv
Rows exported: 30000
File size in bytes: 8551202


### Weak-picks and leakage check

The baseline ranks 30,000 unique content items. Scores range from 0 to 5. A total of 12,452 items have a score of 0, meaning they have no qualifying signals under the current scoring rule. These items should not be treated as high-priority recommendations.

Items with missing CTR or invalid search positions may not qualify for the CTR signal, so their scores can understate uncertainty.

The ranking uses observed update age, CTR, search position, and 90-day impressions. The `trend_direction` outcome is used only in the separate signal audit. Neither `trend_direction` nor `trend_pct` is used to calculate the score. Content and client IDs are used for identification and stable tie-breaking, not as scoring features.

The baseline is a transparent review-priority rule, not a validated prediction model. Before acting on a recommendation, a reviewer should check data completeness, search intent, and business context. Future evaluation should test whether the ranked items produce useful outcomes after the recommended actions are taken.

In [45]:

# Section 3: Check whether the signals are supported by the data.
# trend_direction is used only as an outcome for this audit,
# not as an input to the scoring rule.

audit = df.copy()
audit["is_declining"] = (
    audit["trend_direction"].astype("string").str.lower() == "down"
)

# Signal check 1: stale content vs. recently updated content
refresh_cutoff = df["days_since_last_update"].quantile(0.75)

audit["refresh_bucket"] = "Recently updated / below cutoff"
audit.loc[
    audit["days_since_last_update"].isna(),
    "refresh_bucket"
] = "Unknown update age"
audit.loc[
    audit["days_since_last_update"] >= refresh_cutoff,
    "refresh_bucket"
] = "Stale / top 25% update age"

refresh_table = (
    audit.groupby("refresh_bucket", dropna=False)
    .agg(
        n=("content_id", "size"),
        declining_count=("is_declining", "sum"),
        decline_rate=("is_declining", "mean"),
    )
    .reset_index()
)

print("SIGNAL CHECK 1: Content freshness vs. declining outcome")
display(refresh_table)

# Signal check 2: CTR among content with a valid top-10 position
valid_top10 = (
    audit["avg_position"].notna()
    & (audit["avg_position"] > 0)
    & (audit["avg_position"] <= 10)
    & audit["ctr"].notna()
)

top10_ctr_cutoff = audit.loc[valid_top10, "ctr"].median()

audit["ctr_bucket"] = "Not eligible: missing/position outside top 10"
audit.loc[valid_top10, "ctr_bucket"] = "CTR at/above top-10 median"
audit.loc[
    valid_top10 & (audit["ctr"] < top10_ctr_cutoff),
    "ctr_bucket"
] = "CTR below top-10 median"

ctr_table = (
    audit.groupby("ctr_bucket", dropna=False)
    .agg(
        n=("content_id", "size"),
        declining_count=("is_declining", "sum"),
        decline_rate=("is_declining", "mean"),
    )
    .reset_index()
)

print("\nSIGNAL CHECK 2: CTR vs. search position")
print("Valid top-10 median CTR:", top10_ctr_cutoff)
display(ctr_table)

print("\nInterpretation guide:")
print("- Compare decline_rate between the relevant buckets.")
print("- Higher decline rate in the flagged bucket supports the signal.")
print("- Lower decline rate suggests the opposite.")
print("- Similar rates suggest a mixed or weak relationship.")
print("- Always check n before interpreting a bucket.")


SIGNAL CHECK 1: Content freshness vs. declining outcome


,refresh_bucket,n,declining_count,decline_rate
0,Recently updated / below cutoff,20909,10687,0.51112
1,Stale / top 25% update age,9091,5575,0.613244



SIGNAL CHECK 2: CTR vs. search position
Valid top-10 median CTR: 0.15


,ctr_bucket,n,declining_count,decline_rate
0,CTR at/above top-10 median,6551,3545,0.541139
1,CTR below top-10 median,6432,3766,0.58551
2,Not eligible: missing/position outside top 10,17017,8951,0.526003



Interpretation guide:
- Compare decline_rate between the relevant buckets.
- Higher decline rate in the flagged bucket supports the signal.
- Lower decline rate suggests the opposite.
- Similar rates suggest a mixed or weak relationship.
- Always check n before interpreting a bucket.


### Signal check verdicts

**Signal 1: Content freshness — CONFIRMED**

The stale-content group contains 9,091 items, of which 5,575 are declining (61.32%). The comparison group contains 20,909 items, with 10,687 declining (51.11%). The stale group has a 10.21 percentage-point higher decline rate. This supports the hypothesis that content not updated recently deserves review. However, this is an observational association and does not establish causation.

**Signal 2: Low CTR among top-10 results — CONFIRMED**

Among eligible content with a valid search position in the top 10, 6,432 items have CTR below the group median of 0.15. Their decline rate is 58.55%, compared with 54.11% among the 6,551 items at or above the median. The difference is 4.44 percentage points, providing modest support for the signal. CTR can also be influenced by search intent, query mix, and other factors, so low CTR alone does not prove that a title or snippet needs changing.

**Limitations:** These checks compare observed decline rates and do not prove that either action causes better future performance. The `trend_direction` field is used only as the audit outcome and is not an input to the scoring rule.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [46]:

# Section 4: Review the top 20 ranked items

top20 = scored.head(20).copy()

def explain_why(row):
    reasons = []

    if pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] >= refresh_threshold:
        reasons.append("Update age is in the oldest 25% of the dataset")

    if (
        pd.notna(row["avg_position"])
        and 0 < row["avg_position"] <= 10
        and pd.notna(row["ctr"])
        and row["ctr"] < top10_ctr_median
    ):
        reasons.append("CTR is below the median for valid top-10 results")

    if pd.notna(row["impressions_90d"]) and row["impressions_90d"] >= volume_threshold:
        reasons.append("Impressions are in the highest 25% of the dataset")

    return "; ".join(reasons) if reasons else "No qualifying signal; investigate the available data"

top20["why_selected"] = top20.apply(explain_why, axis=1)

top20["what_could_make_it_wrong"] = (
    "The signal may not reflect a real improvement opportunity; "
    "check search intent, data completeness, and business context before acting."
)

review_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action_label",
    "why_selected",
    "what_could_make_it_wrong",
]

display(top20[review_columns])
print("Items reviewed:", len(top20))


,rank,content_id,score,reason_code,action_label,why_selected,what_could_make_it_wrong
0,1,content_5fe46e04994d,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
1,2,content_36ff89c8214e,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
2,3,content_c8e9d6ab9013,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
3,4,content_a7427266c305,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
4,5,content_91652435f57a,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
5,6,content_f42eb861c6dd,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
6,7,content_97a86caf3a3d,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
7,8,content_c1fe78bc4e37,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
8,9,content_4a6607efcb46,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...
9,10,content_4c76e9b13aea,5,CTR_REVIEW,Review title and snippet,Update age is in the oldest 25% of the dataset...,The signal may not reflect a real improvement ...


Items reviewed: 20


In [47]:

# Final section: Weak-picks and leakage check

# 1. Check whether the ranking contains tied scores.
print("Score distribution:")
print(scored["score"].value_counts().sort_index(ascending=False))

# 2. Inspect low-scoring items as possible weak picks.
print("\nLowest-scoring items:")
display(
    scored.tail(10)[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action_label",
            "impressions_90d",
            "days_since_last_update",
            "avg_position",
            "ctr",
        ]
    ]
)

# 3. Verify output integrity.
print("\nIntegrity checks:")
print("Total rows:", len(scored))
print("Unique content IDs:", scored["content_id"].nunique())
print("Missing scores:", scored["score"].isna().sum())
print("Missing reason codes:", scored["reason_code"].isna().sum())
print("Missing action labels:", scored["action_label"].isna().sum())
print("Ranks are unique:", scored["rank"].is_unique)

# 4. Leakage check: the scoring rule must not use outcome-derived fields.
score_inputs = [
    "days_since_last_update",
    "avg_position",
    "ctr",
    "impressions_90d",
]

forbidden_inputs = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

print("\nScoring inputs:", score_inputs)
print("Forbidden outcome-derived inputs:", forbidden_inputs)
print("Leakage check: score was constructed only from the listed scoring inputs.")
print("Outcome fields are used only for the separate signal audit.")
print("\nCSV exists:", output_path.exists())
print("CSV size (bytes):", output_path.stat().st_size if output_path.exists() else 0)


Score distribution:
score
5      474
4     1175
3     3352
2     8873
1     3674
0    12452
Name: count, dtype: int64

Lowest-scoring items:


,rank,content_id,score,reason_code,action_label,impressions_90d,days_since_last_update,avg_position,ctr
29990,29991,content_fc1213c8f132,0,NO_STRONG_SIGNAL,Manual review if needed,1,98,15.0,0.0
29991,29992,content_fc5bdea3ae5f,0,NO_STRONG_SIGNAL,Manual review if needed,1,20,0.0,0.0
29992,29993,content_fd00ef7f3d07,0,NO_STRONG_SIGNAL,Manual review if needed,1,20,0.0,0.0
29993,29994,content_fd47b1a95305,0,NO_STRONG_SIGNAL,Manual review if needed,1,20,0.0,0.0
29994,29995,content_fd690dc28977,0,NO_STRONG_SIGNAL,Manual review if needed,1,20,0.0,0.0
29995,29996,content_fda85f089ea9,0,NO_STRONG_SIGNAL,Manual review if needed,1,20,0.0,0.0
29996,29997,content_feb706a6b155,0,NO_STRONG_SIGNAL,Manual review if needed,1,20,0.0,0.0
29997,29998,content_fee223be84c1,0,NO_STRONG_SIGNAL,Manual review if needed,1,20,0.0,0.0
29998,29999,content_ff4c72980a9e,0,NO_STRONG_SIGNAL,Manual review if needed,1,20,0.0,0.0
29999,30000,content_ffffc9a0c08c,0,NO_STRONG_SIGNAL,Manual review if needed,1,8,0.0,0.0



Integrity checks:
Total rows: 30000
Unique content IDs: 30000
Missing scores: 0
Missing reason codes: 0
Missing action labels: 0
Ranks are unique: True

Scoring inputs: ['days_since_last_update', 'avg_position', 'ctr', 'impressions_90d']
Forbidden outcome-derived inputs: ['trend_direction', 'trend_pct', 'is_declining_label']
Leakage check: score was constructed only from the listed scoring inputs.
Outcome fields are used only for the separate signal audit.

CSV exists: True
CSV size (bytes): 8551202


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [48]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Weak-picks and leakage check

The baseline ranks 30,000 unique content items. Scores range from 0 to 5. A total of 12,452 items have a score of 0, meaning they have no qualifying signals under the current scoring rule. These items should not be treated as high-priority recommendations.

Items with missing CTR or invalid search positions may not qualify for the CTR signal, so their scores can understate uncertainty.

The ranking uses observed update age, CTR, search position, and 90-day impressions. The `trend_direction` outcome is used only in the separate signal audit. Neither `trend_direction` nor `trend_pct` is used to calculate the score. Content and client IDs are used for identification and stable tie-breaking, not as scoring features.

The baseline is a transparent review-priority rule, not a validated prediction model. Before acting on a recommendation, a reviewer should check data completeness, search intent, and business context. Future evaluation should test whether the ranked items produce useful outcomes after the recommended actions are taken.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [49]:
import os
os.makedirs("work/outputs", exist_ok=True)

In [50]:
import os
print(os.path.exists("work/outputs/baseline_action_score.csv"))

True
